# Exercise 5: Extending a tokeniser

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Pretrained Language Models · version 0.1.0*

> Extend a tokeniser without changing old IDs. Verify new embedding gradients and save/reload equivalence, then test whether reduced fertility improves downstream performance.

**Setup.** `pip install torch transformers peft scikit-learn psutil requests "turkicnlp==0.5.0"`. Models are loaded at the Hugging Face revisions pinned in `MODELS` below, from a local Hugging Face cache (`HF_HUB_DIR`, default `~/.cache/huggingface/hub`), from a folder of copies (`MODELS_DIR`) or from the Hub. Data come from commit-pinned GitHub repositories: SIB-200 topic classification (seven topics; FLORES-200 sentences, so the same 1,004 sentences exist in every language), Kardeş-NLU's Kazakh XCOPA and the BERTurk repository's published evaluations. Everything runs on the CPU. Training results are cached per configuration in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), so an interrupted run resumes; record elapsed time and thread count for each run.

The new tokens are whole Kazakh words from the training split, added as `AddedToken`s that match only whole words. The downstream comparison repeats Exercise 2's LoRA rank-8 protocol with three seeds, training the new embedding rows together with the adapters.

In [ ]:
import collections, csv, hashlib, io, json, math, os, random, re, statistics, threading, time
from pathlib import Path
import requests

SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
KARDES = "https://raw.githubusercontent.com/lksenel/Kardes-NLU/48eea77fda904e731d228de833f4c57d130fad9c/Data/"
BERTURK = "https://raw.githubusercontent.com/stefan-it/turkish-bert/0ab49e288d07f0aa40bcc1c28b655c80c1350fee/"
MODELS = {   # Hugging Face model -> pinned revision
    "cis-lmu/glot500-base": "d4d7c1ec01828fdf7452a4ccf7b55177aced175e",
    "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "c2e19453aae46f98a865abfdf173f35f2e119f5a",
    "meta-llama/Meta-Llama-3-8B": "8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920",
    "allenai/Olmo-3-1025-7B": "0a15c289b17ad4f27aabc863276ec525d5312c01",
    "Qwen/Qwen3.6-27B": "6a9e13bd6fc8f0983b9b99948120bc37f49c13e9",
    "Qwen/Qwen2.5-1.5B-Instruct": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
HF_HUB_DIR = Path(os.environ.get("HF_HUB_DIR", "~/.cache/huggingface/hub")).expanduser()
MODELS_DIR = Path(os.environ.get("MODELS_DIR", CACHE / "models"))
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def model_source(name):
    """(path or name, revision) for from_pretrained: a local snapshot of the pinned revision if present."""
    rev = MODELS[name]
    for p in (HF_HUB_DIR / f"models--{name.replace('/', '--')}" / "snapshots" / rev,
              MODELS_DIR / f"models--{name.replace('/', '--')}--{rev}"):
        if (p / "config.json").exists():
            return str(p), None
    return name, rev

def tokenizer(name):
    from transformers import AutoTokenizer
    path, rev = model_source(name)
    return AutoTokenizer.from_pretrained(path, revision=rev)

def sib(lang, split):
    """SIB-200 rows (index_id, topic, text) for a FLORES language code such as kaz_Cyrl."""
    text = fetch(f"{SIB}{lang}/{split}.tsv").read_text(encoding="utf-8")
    return [(r["index_id"], r["category"], r["text"]) for r in csv.DictReader(io.StringIO(text), delimiter="\t")]

LABELS = ["entertainment", "geography", "health", "politics", "science/technology", "sports", "travel"]
GLOT = "cis-lmu/glot500-base"

_CTS = {}
def to_cts(lang, text):
    """Common Turkic Alphabet version of a sentence (TurkicNLP transliterator); Turkish is unchanged."""
    if lang not in _CTS:
        from turkicnlp.scripts import Script
        from turkicnlp.scripts.transliterator import Transliterator
        src = {"tur": Script.LATIN}.get(lang, Script.CYRILLIC)
        _CTS[lang] = Transliterator(lang, src, Script.COMMON_TURKIC)
    return _CTS[lang].transliterate(text)

def macro_f1(gold, pred):
    from sklearn.metrics import f1_score
    return f1_score(gold, pred, labels=list(range(len(LABELS))), average="macro", zero_division=0)

class PeakRSS:
    """Peak resident memory of this process while the block runs (sampled every 0.2 s)."""
    def __enter__(self):
        import psutil
        self.p, self.peak, self.stop = psutil.Process(), 0, False
        def sample():
            while not self.stop:
                self.peak = max(self.peak, self.p.memory_info().rss); time.sleep(0.2)
        self.t = threading.Thread(target=sample, daemon=True); self.t.start(); return self
    def __exit__(self, *exc):
        self.stop = True; self.t.join()

def run(mode="lora", r=8, train="kaz_Cyrl", script="native", seed=1, epochs=10, new_tokens=0,
        tests=(("kaz_Cyrl", "native"),)):
    """Fine-tune Glot500-base for SIB-200 topics; select the epoch on the training language's dev
    macro-F1; return test macro-F1/accuracy for each (language, script) in `tests`. Cached on disk.
    mode: "head" (classifier only), "lora" (query/value, rank r, alpha 2r, classifier trained),
    "full" (all weights except the word embeddings). new_tokens > 0 adds that many Kazakh words to
    the tokenizer (see add_kazakh_words) and trains their embedding rows together with LoRA."""
    cfg = dict(base=GLOT, rev=MODELS[GLOT], mode=mode, r=r if mode == "lora" else None, train=train,
               script=script, seed=seed, epochs=epochs, new_tokens=new_tokens, tests=[list(t) for t in tests],
               batch=16, lr={"head": 1e-3, "lora": 2e-4, "full": 2e-5}[mode], warmup=0.1, v=1)
    key = hashlib.sha1(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:16]
    path = CACHE / "ch12-runs" / f"{key}.json"
    if path.exists():
        return json.loads(path.read_text())
    import torch
    from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    torch.manual_seed(seed); torch.set_num_threads(THREADS)
    src, rev = model_source(GLOT)
    tok = tokenizer(GLOT)
    model = AutoModelForSequenceClassification.from_pretrained(
        src, revision=rev, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)),
        label2id={l: i for i, l in enumerate(LABELS)})
    added = add_kazakh_words(tok, model, new_tokens) if new_tokens else []
    if mode == "full":
        model.roberta.embeddings.word_embeddings.weight.requires_grad_(False)
    elif mode == "head":
        for n, p in model.named_parameters():
            p.requires_grad_(n.startswith("classifier"))
    else:
        from peft import LoraConfig, TaskType, get_peft_model
        extra = {"trainable_token_indices": {"word_embeddings": added}} if added else {}
        model = get_peft_model(model, LoraConfig(task_type=TaskType.SEQ_CLS, r=r, lora_alpha=2 * r, lora_dropout=0.1,
                                                 target_modules=["query", "value"], modules_to_save=["classifier"], **extra))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    def data(lang, split, scr):
        rows = sib(lang, split)
        texts = [t if scr == "native" else to_cts(lang.split("_")[0], t) for _, _, t in rows]
        return texts, [LABELS.index(c) for _, c, _ in rows]
    def predict(texts):
        model.eval(); out = []
        with torch.inference_mode():
            for i in range(0, len(texts), 32):
                enc = tok(texts[i:i + 32], padding=True, truncation=True, max_length=256, return_tensors="pt")
                out += model(**enc).logits.argmax(-1).tolist()
        return out
    tr_x, tr_y = data(train, "train", script)
    dev_x, dev_y = data(train, "dev", script)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=0.01)
    steps = epochs * math.ceil(len(tr_x) / cfg["batch"])
    sched = get_linear_schedule_with_warmup(opt, int(cfg["warmup"] * steps), steps)
    rng = random.Random(seed); best = (-1, None, None); history = []
    t0 = time.time()
    with PeakRSS() as mem:
        for ep in range(1, epochs + 1):
            model.train(); order = list(range(len(tr_x))); rng.shuffle(order)
            for i in range(0, len(order), cfg["batch"]):
                b = order[i:i + cfg["batch"]]
                enc = tok([tr_x[j] for j in b], padding=True, truncation=True, max_length=256, return_tensors="pt")
                loss = model(**enc, labels=torch.tensor([tr_y[j] for j in b])).loss
                loss.backward(); opt.step(); sched.step(); opt.zero_grad()
            f1 = macro_f1(dev_y, predict(dev_x)); history.append(round(f1, 4))
            if f1 > best[0]:
                best = (f1, ep, {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad})
        with torch.no_grad():
            for n, p in model.named_parameters():
                if n in best[2]:
                    p.copy_(best[2][n])
        results = {}
        for lang, scr in tests:
            x, y = data(lang, "test", scr); pred = predict(x)
            results[f"{lang}/{scr}"] = {"macro_f1": macro_f1(y, pred), "accuracy": sum(a == b for a, b in zip(y, pred)) / len(y),
                                        "n": len(y), "pred": pred}
    res = dict(cfg=cfg, trainable=trainable, total=total, dev_history=history, best_epoch=best[1],
               dev_macro_f1=best[0], tests=results, seconds=time.time() - t0, peak_rss_gb=mem.peak / 1e9,
               added_tokens=added)
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(res))
    return res

PROBE = r"""
import json, sys, threading, time, psutil, torch
a = json.loads(sys.argv[1]); torch.set_num_threads(a["threads"]); torch.manual_seed(0)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
proc = psutil.Process(); start = proc.memory_info().rss
tok = AutoTokenizer.from_pretrained(a["path"]); m = AutoModelForSequenceClassification.from_pretrained(a["path"], num_labels=7)
if a["mode"] == "full":
    m.roberta.embeddings.word_embeddings.weight.requires_grad_(False)
elif a["mode"] == "head":
    for n, p in m.named_parameters(): p.requires_grad_(n.startswith("classifier"))
else:
    from peft import LoraConfig, TaskType, get_peft_model
    m = get_peft_model(m, LoraConfig(task_type=TaskType.SEQ_CLS, r=a["r"], lora_alpha=2 * a["r"], lora_dropout=0.1,
                                     target_modules=["query", "value"], modules_to_save=["classifier"]))
loaded = proc.memory_info().rss; peak = [loaded]; stop = []
def sample():
    while not stop: peak[0] = max(peak[0], proc.memory_info().rss); time.sleep(0.05)
t = threading.Thread(target=sample, daemon=True); t.start()
opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=1e-4)
enc = tok(a["texts"], padding=True, return_tensors="pt"); y = torch.zeros(len(a["texts"]), dtype=torch.long)
t0 = time.time()
for _ in range(a["steps"]):
    m(**enc, labels=y).loss.backward(); opt.step(); opt.zero_grad()
sec = (time.time() - t0) / a["steps"]; stop.append(1); t.join()
print(json.dumps({"start": start, "loaded": loaded, "peak": peak[0], "sec_per_step": sec}))
"""

def peak_memory(mode, r=8, steps=5):
    """Peak resident memory of a fresh process that loads Glot500 and takes `steps` training steps
    on the first 16 Kazakh training sentences (batch 16), and its seconds per step. Cached."""
    import subprocess, sys
    key = f"{mode}-{r if mode == 'lora' else ''}-{steps}-{THREADS}"
    path = CACHE / "ch12-memory.json"
    saved = json.loads(path.read_text()) if path.exists() else {}
    if key not in saved:
        texts = [t for _, _, t in sib("kaz_Cyrl", "train")[:16]]
        args = dict(path=model_source(GLOT)[0], mode=mode, r=r, steps=steps, threads=THREADS, texts=texts)
        out = subprocess.run([sys.executable, "-c", PROBE, json.dumps(args, ensure_ascii=False)],
                             capture_output=True, text=True, check=True).stdout.strip().splitlines()[-1]
        saved[key] = json.loads(out); path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(saved))
    return saved[key]

def kazakh_candidates(tok, n):
    """The n most frequent Kazakh word forms of the SIB-200 training split that Glot500 splits into
    three or more pieces (ties broken alphabetically)."""
    words = collections.Counter(w for _, _, t in sib("kaz_Cyrl", "train") for w in re.findall(r"\w+", t))
    long = [w for w, c in sorted(words.items(), key=lambda x: (-x[1], x[0]))
            if len(tok.tokenize(" " + w)) >= 3]
    return long[:n]

def add_kazakh_words(tok, model, n):
    """Add whole-word tokens without changing old IDs; initialise each new row as the mean of the
    old embeddings of the word's pieces. Returns the new IDs."""
    import torch
    from tokenizers import AddedToken
    words = kazakh_candidates(tok, n)
    old_len = len(tok)
    pieces = {w: tok(" " + w, add_special_tokens=False)["input_ids"] for w in words}
    tok.add_tokens([AddedToken(w, single_word=True, lstrip=True) for w in words])
    model.resize_token_embeddings(len(tok), mean_resizing=False)
    emb = model.get_input_embeddings().weight
    with torch.no_grad():
        for w in words:
            emb[tok.convert_tokens_to_ids(w)] = emb[pieces[w]].mean(0)
    return list(range(old_len, len(tok)))

def wilson(k, n, z=1.959963984540054):
    p = k / n; den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h

print("models:", ", ".join(f"{n} ({'local' if model_source(n)[1] is None else 'Hub'})" for n in MODELS))

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification
N = 300
base_tok, tok = tokenizer(GLOT), tokenizer(GLOT)
src, rev = model_source(GLOT)
torch.manual_seed(0)
model = AutoModelForSequenceClassification.from_pretrained(src, revision=rev, num_labels=len(LABELS))
old_vocab = tok.get_vocab()
new_ids = add_kazakh_words(tok, model, N)
print(f"added {len(new_ids)} Kazakh words, e.g. {tok.convert_ids_to_tokens(new_ids[:6])}")
new_vocab = tok.get_vocab()
print("old IDs unchanged:", all(new_vocab[p] == i for p, i in old_vocab.items()),
      f"| vocabulary {len(old_vocab):,} -> {len(tok):,} | embedding rows {model.get_input_embeddings().weight.shape[0]:,}")
test = [t for _, _, t in sib("kaz_Cyrl", "test")]
other = [t for _, _, t in sib("tur_Latn", "test")]
n_old = sum(len(base_tok.encode(s, add_special_tokens=False)) for s in test)
n_new = sum(len(tok.encode(s, add_special_tokens=False)) for s in test)
print(f"Kazakh test tokens {n_old:,} -> {n_new:,} ({n_new / n_old - 1:+.1%}); Turkish test encodings unchanged: "
      f"{all(base_tok.encode(s) == tok.encode(s) for s in other)}")
del model                                            # one Glot500 copy in memory at a time
s = test[0]; print("round trip:", tok.decode(tok.encode(s, add_special_tokens=False)) == s, "|", " ".join(tok.tokenize(s)[:14]))

# Gradient check: which embedding rows receive gradients under each policy?
from peft import LoraConfig, TaskType, get_peft_model
enc = tok(test[:8], padding=True, return_tensors="pt"); y = torch.zeros(8, dtype=torch.long)
used = sorted(set(enc["input_ids"].flatten().tolist()) & set(new_ids))
for policy in ("attention LoRA only", "LoRA + trainable new rows"):
    torch.manual_seed(0)
    m = AutoModelForSequenceClassification.from_pretrained(src, revision=rev, num_labels=len(LABELS))
    add_kazakh_words(tokenizer(GLOT), m, N)
    extra = {"trainable_token_indices": {"word_embeddings": new_ids}} if "rows" in policy else {}
    m = get_peft_model(m, LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, target_modules=["query", "value"],
                                     modules_to_save=["classifier"], **extra))
    m(**enc, labels=y).loss.backward()
    grads = {n: p.grad for n, p in m.named_parameters() if p.requires_grad and "word_embeddings" in n}
    g = next(iter(grads.values()), None)
    print(f"  {policy:<26}: trainable embedding tensors {len(grads)}"
          + (f", shape {tuple(g.shape)}, rows with nonzero gradient {int((g.abs().sum(-1) > 0).sum())} "
             f"(new words in the batch: {len(used)})" if g is not None else " -> new rows cannot learn"))
    if "rows" in policy:
        out = CACHE / "ch12-ext-adapter"; m.save_pretrained(out); tok.save_pretrained(out)
        from peft import PeftModel
        m2 = AutoModelForSequenceClassification.from_pretrained(src, revision=rev, num_labels=len(LABELS))
        m2.resize_token_embeddings(len(tok), mean_resizing=False); m2 = PeftModel.from_pretrained(m2, out)
        m.eval(); m2.eval()
        with torch.inference_mode():
            same = torch.allclose(m(**enc).logits, m2(**enc).logits, atol=1e-5)
        print(f"  save/reload with a freshly resized base: logits equal = {same}; adapter file "
              f"{(out / 'adapter_model.safetensors').stat().st_size / 1e6:.0f} MB (PEFT saves the whole resized embedding matrix)")
        del m2
    del m

# Downstream: SIB-200 Kazakh with and without the extension (LoRA r=8, three seeds, same protocol as Exercise 2).
base = [run(seed=s) for s in (1, 2, 3)]
ext = [run(seed=s, new_tokens=N) for s in (1, 2, 3)]
for name, rs in (("baseline", base), (f"+{N} words", ext)):
    f = [r["tests"]["kaz_Cyrl/native"]["macro_f1"] for r in rs]
    print(f"  {name:<11} test macro-F1 " + "  ".join(f"{x:.3f}" for x in f) + f"   mean {statistics.mean(f):.3f}"
          f"   trainable {rs[0]['trainable']:,}")

## Worked solution

Output of `experiments/ch12_exercises/run_ch12_exercises.py`:

```
added 300 Kazakh words, e.g. ['ықтимал', 'айтарлықтай', 'жаяу', 'толығымен', 'қамтиды', 'қамтитын']
old IDs unchanged: True | vocabulary 401,145 -> 401,445 | embedding rows 401,445
Kazakh test tokens 6,735 -> 6,569 (-2.5%); Turkish test encodings unchanged: True
round trip: True | ▁Мут ация ▁жаңа ▁ген е тикалық ▁вари ация ны ▁қоса ды ▁және ▁селекци я
  attention LoRA only       : trainable embedding tensors 0 -> new rows cannot learn
  LoRA + trainable new rows : trainable embedding tensors 1, shape (300, 768), rows with nonzero gradient 1 (new words in the batch: 1)
  save/reload with a freshly resized base: logits equal = True; adapter file 1238 MB (PEFT saves the whole resized embedding matrix)
  baseline    test macro-F1 0.807  0.861  0.843   mean 0.837   trainable 890,887
  +300 words  test macro-F1 0.822  0.837  0.794   mean 0.818   trainable 1,121,287
```

**Extension checks.** The code adds 300 frequent training words that the original tokeniser splits into at least three pieces. It preserves old IDs and initialises new rows from the mean of their old pieces. The checked Turkish test encodings remain unchanged; this is a sample-specific result. The reload probe reproduces the checked logits.

**Gradients and saving.** Attention-only LoRA leaves embeddings frozen. With `trainable_token_indices`, the added rows become trainable; the probe confirms a nonzero gradient for the one new word present in its batch. In the recorded PEFT version and configuration, saving the resized model writes a 1,238 MB adapter file containing the full embedding matrix. A smaller custom format needs its own reconstruction and reload checks.

**Task results.** The extension reduces Kazakh test tokens from 6,735 to 6,569 (2.5%). Mean macro-F1 is 0.818 with the extension and 0.837 without it, across three seeds. The individual changes differ in sign. This run does not demonstrate an accuracy gain, but it also does not establish why the mean fell or whether more training would help. The runner uses the same 256-token truncation policy as Exercise 2; audit the length effects as well as vocabulary changes.

## Your turn — extension

1. Initialise the new rows randomly instead of from their pieces and repeat the downstream comparison.
2. Add 3,000 words instead of 300 and measure fertility and macro-F1.